# 视觉引导机械臂抓取系统

## 项目简介

本项目实现一个由视觉侧负责目标识别与坐标换算、Arduino 侧负责动作执行的机械臂抓取系统。上位机先检测目标并转换为抓取位姿参数，再通过串口把抓取目标的底座角、主臂角、前臂角和投放槽位发送给 Arduino，机械臂据此执行完整抓取流程。


## 主要器件

| 器件 | 数量 | 说明 |
| --- | --- | --- |
| PC / Jetson Nano | 1 | 视觉识别与坐标转换 |
| 摄像头 | 1 | 检测抓取目标 |
| Arduino Uno | 1 | 接收位姿并控制机械臂 |
| 舵机 | 4 | 底座、主臂、前臂、夹爪 |
| 机械臂结构 | 1 套 | 完成抓取与放置 |


## 核心功能

- 视觉端识别目标并计算抓取位姿。
- 上位机通过串口下发抓取命令。
- Arduino 按动作序列执行预抓取、下探、夹取、抬升与放置。
- 按投放槽位选择不同的底座放置角。
- 支持 `HOME` 指令快速复位。


## 引脚分配

| 模块 | 引脚 |
| --- | --- |
| 底座舵机 | D3 |
| 主臂舵机 | D5 |
| 前臂舵机 | D6 |
| 夹爪舵机 | D9 |
| 上位机串口通信 | USB Serial |


## 接线说明

- 四路舵机信号线分别接 D3、D5、D6、D9，建议使用独立 5V 电源供电。
- 舵机电源地线必须与 Arduino 共地，否则串口正常但动作会抖动或失控。
- 摄像头接在上位机端，负责获取目标图像，不直接接到 Arduino。
- 机械臂装配完成后应先验证 `HOME` 姿态，确保复位过程不会发生关节干涉。


## 串口协议

- 复位命令：`HOME`。
- 抓取命令：`G,base,shoulder,elbow,slot`。
- 示例：`G,102,118,134,2` 表示抓取目标位姿为底座 102 度、主臂 118 度、前臂 134 度，投放到 2 号槽位。
- Arduino 会回传 `ACK,START`、`ACK,DONE` 或 `BUSY` 作为执行状态反馈。


## 运行方式

1. 打开 `src/vision_guided_robotic_arm_grasping_system/vision_guided_robotic_arm_grasping_system.ino` 并上传。
2. 上位机端完成目标识别、像素坐标到关节角的换算。
3. 先发送 `HOME` 验证复位姿态，再发送抓取命令测试动作链。
4. 根据机械臂结构微调各个安全角度与放置槽位角度。
5. 加入限位或软限位后再进行连续抓取测试。


## 控制逻辑说明

- `parseCommand()` 负责解析 `HOME` 与 `G,base,shoulder,elbow,slot` 两类命令。
- `executeGrasp()` 对抓取角度先做范围约束，防止视觉端给出超界位姿。
- 动作流程先张开夹爪并移动到预抓取位，再下降到抓取位，降低直接碰撞目标的风险。
- 夹紧目标后先抬起，再旋转到底座投放角，最后松爪放置并回原点。
- `busyExecuting` 用于防止上一个动作尚未结束时重复接收新抓取指令。


## 验证标准

| 测试项 | 通过标准 |
| --- | --- |
| 协议解析 | 串口发送命令后能收到 ACK |
| 抓取流程 | 机械臂能完成预抓取、夹取、抬升 |
| 分类放置 | 不同 `slot` 会对应不同投放位置 |
| 异常互斥 | 忙碌期间再次下发命令会返回 `BUSY` |


## 可扩展方向

- 增加相机标定与更精确的坐标变换模型。
- 增加逆运动学求解，减少对上位机直接下发角度的依赖。
- 增加多类别分拣与托盘管理逻辑。
- 增加抓取失败检测与自动重试机制。
